# Day 11: LangChain Basics (Part 1)

## Goal

Set up LangChain with OpenAI or Ollama and build simple chains.

By the end of this notebook you will have:

- a LangChain model setup for OpenAI or Ollama
- a simple prompt-to-response chain
- a second chain that uses the output of the first chain
- a working foundation for later RAG notebooks

This notebook is intentionally simple. The focus is the LangChain flow, not advanced orchestration.

## Step 0: Sync the Project Once

The Day 11 packages are already included in `pyproject.toml`, so learners only need:

```bash
uv sync
```

Set the values you want in `.env`.

Example:

```text
OPENAI_API_KEY=your_openai_key
OPENAI_MODEL=your_openai_model
OLLAMA_MODEL=llama3.2
OLLAMA_BASE_URL=http://localhost:11434
```

If you plan to use Ollama, make sure the Ollama server is running locally and that the model is already pulled.

For course maintainers, the Day 11 packages were added with:

```bash
uv add langchain langchain-openai langchain-ollama
```

In [ ]:
import os

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

print("Imports loaded successfully.")


In [ ]:
load_dotenv(override=True)

config_summary = {
    "OPENAI_API_KEY_present": bool(os.getenv("OPENAI_API_KEY")),
    "OPENAI_MODEL": os.getenv("OPENAI_MODEL"),
    "OLLAMA_MODEL": os.getenv("OLLAMA_MODEL"),
    "OLLAMA_BASE_URL": os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
}

config_summary

## Step 1: Create a Model Helper

This lets us switch between OpenAI and Ollama without changing the rest of the chain code.

In [ ]:
def get_llm(provider: str):
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        model = os.getenv("OPENAI_MODEL")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is not set in .env")
        if not model:
            raise ValueError("OPENAI_MODEL is not set in .env")
        return ChatOpenAI(model=model, api_key=api_key)

    if provider == "ollama":
        model = os.getenv("OLLAMA_MODEL")
        if not model:
            raise ValueError("OLLAMA_MODEL is not set in .env")
        return ChatOllama(
            model=model,
            base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
        )

    raise ValueError("provider must be 'openai' or 'ollama'")


print("Model helper ready.")

## Step 2: Build a Simple Prompt + Model Flow

We will:
- create a prompt
- send it to the model
- print the response


In [ ]:
provider = "openai"  # Change to "ollama" if you want to use Ollama.
llm = get_llm(provider)

explain_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a clear and practical trainer for software engineers."),
        ("human", "Explain {topic} in 3 short bullet points for beginners."),
    ]
)


def explain_topic(topic: str) -> str:
    messages = explain_prompt.format_messages(topic=topic)
    response = llm.invoke(messages)
    return response.content


print("Prompt and helper ready.")


In [ ]:
topic = "embeddings"
explanation_text = explain_topic(topic)

print(explanation_text)


## Step 3: Build a Second Prompt from the First Output

Now we take the first explanation and feed it into another prompt.

This shows how LangChain can connect steps together.


In [ ]:
quiz_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You create short learning exercises for engineers."),
        ("human", """Using the explanation below, create 3 quiz questions.

{explanation}"""),
    ]
)


def create_quiz(explanation: str) -> str:
    messages = quiz_prompt.format_messages(explanation=explanation)
    response = llm.invoke(messages)
    return response.content


print("Quiz prompt and helper ready.")


In [ ]:
quiz_text = create_quiz(explanation_text)

print(quiz_text)


## Step 4: Try a Different Topic

Change the topic below and run the next cell again.

Good examples:

- `vector databases`
- `tokenization`
- `FastAPI`
- `RAG`

In [ ]:
topic = "vector databases"
explanation_text = explain_topic(topic)
quiz_text = create_quiz(explanation_text)

print(quiz_text)


## Day 11 Recap

You now have a working LangChain setup with:

- provider selection for OpenAI or Ollama
- a simple prompt + model flow
- a second prompt that uses the first output

This is enough to understand the basic LangChain mental model before adding retrieval.
